# Script 17 — Marker-level associations of rhythm with the components of KDM biological age and SII (Supp Table S16)

Regresses each of the ten KDM biological-age markers, and the three inputs of
the systemic immune-inflammation index (SII), on each rhythm metric under the
core covariates, within each age cell (18-59, 60+, and 18+ as a secondary
pooled cell). KDM-BA and SII share inputs: KDM-BA includes lymphocyte
percentage (`LBXLYPCT`) and white cell count (`LBXWBCSI`), and SII is
platelets × neutrophils / lymphocytes. The table shows which markers move with
rhythm and whether the immune markers are among them. It is a descriptive
decomposition, not a mediation model.

Method: survey-weighted least squares with HC3 standard errors, matching
scripts 7 and 8. Exposure and marker are both z-standardised on the
survey-weighted mean and SD of the estimation sample, so coefficients are per
1 SD and comparable across markers. Holm correction across the 13 fitted
markers within each rhythm metric × age cell. Weights enter the point estimate
only; the variance carries no stratum or PSU adjustment, so the standard
errors are smaller than design-based ones.

SII identity: because SII = platelets × neutrophils / lymphocytes,

$$\log \mathrm{SII} = \log(\mathrm{PLT}) + \log(\mathrm{NEU}) - \log(\mathrm{LYM})$$

so the coefficient of any predictor on log SII equals that combination of its
coefficients on the three logged components, fitted on the same sample with
the same covariates. Step 3 raises if the discrepancy exceeds 1e-6.

Reads `data/processed/8_nhanes_analytic.csv` and checks the marker mapping
against `scripts/5_5_compute_kdm_ba.R`.
Writes `result/supplementary/SuppTableS16_KDM_Component_Specificity.{xlsx,csv}`.

## Step 0 — Imports, paths, marker mapping

In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.options.display.max_columns = 60

def find_root():
    """Locate the project root without assuming where the kernel started."""
    env = os.environ.get('NHANES_SEM_ROOT')
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for q in cands:
        if (q / 'data' / 'processed').is_dir() and (q / 'scripts').is_dir():
            return q
    raise RuntimeError(
        'cannot find the project root from ' + str(Path.cwd())
        + ' -- start Jupyter inside the project, or set NHANES_SEM_ROOT')

BASE     = find_root()
ANALYTIC = BASE / 'data' / 'processed' / '8_nhanes_analytic.csv'
SUPP     = BASE / 'result' / 'supplementary'
SRC      = BASE / 'figure' / 'source_data'
for q in (SUPP, SRC):
    q.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BASE / 'scripts'))

# Reload project modules so the notebook runs against the code on disk.
import importlib
import nature_palette, wls_utils, mplus_parse, mplus_export
for _m in (nature_palette, wls_utils, mplus_parse, mplus_export,):
    importlib.reload(_m)

from wls_utils import wls_fit
from mplus_export import holm
from nature_palette import EXP_ORDER, AGE_LABEL, fmt_est, fmt_ci

# NHANES code -> (name in the KDM panel, marker group), copied from
# 5_5_compute_kdm_ba.R and checked against it at run time.
KDM_MARKERS = [
    ('LBXSAL',   'albumin',      'KDM panel: metabolic/organ'),
    ('LBXLYPCT', 'lymph',        'KDM panel: immune'),
    ('LBXMCVSI', 'mcv',          'KDM panel: haematological'),
    ('LBXSGL',   'glucose_mmol', 'KDM panel: metabolic/organ'),
    ('LBXRDW',   'rdw',          'KDM panel: haematological'),
    ('LBXSCR',   'creat',        'KDM panel: metabolic/organ'),
    ('LBXSAPSI', 'alp',          'KDM panel: metabolic/organ'),
    ('LBXWBCSI', 'wbc',          'KDM panel: immune'),
    ('LBXGH',    'hba1c',        'KDM panel: metabolic/organ'),
    ('LBXTC',    'totchol',      'KDM panel: metabolic/organ'),
]
# SII's inputs. Lymphocytes appear in the KDM panel as a percentage and here as
# a count; the two are related but not the same variable, so both are fitted.
SII_INPUTS = [
    ('LBDNENO',  'neutrophil count', 'SII input (not in KDM panel)'),
    ('LBDLYMNO', 'lymphocyte count', 'SII input (lymph % is in KDM panel)'),
    ('LBXPLTSI', 'platelet count',   'SII input (not in KDM panel)'),
]

# Same rename and covariate set as scripts 7 and 8. The analytic file is
# already cleaned upstream (education codes 1-5 only, sleep hours capped), so
# refused/don't-know codes need no handling here.
RENAME = {'RIDAGEYR': 'AGE', 'RIAGENDR': 'SEX', 'BMXBMI': 'BMI',
          'DMDEDUC2': 'EDU', 'Wear_Time': 'WEAR_T', 'SLD010H': 'SLEEP_H'}
COVS = ['AGE', 'SEX', 'BMI', 'EDU', 'WEAR_T', 'SLEEP_H']
EXPOSURES = list(EXP_ORDER)   # RA, IS, IV
CELLS = [('18-59', 18, 60), ('60+', 60, 999), ('18+', 18, 999)]

print('root     ', BASE)
print('analytic ', ANALYTIC.name, '-- exists:', ANALYTIC.exists())
print('markers  ', len(KDM_MARKERS), 'KDM panel +', len(SII_INPUTS), 'SII inputs')

## Step 1 — Load, and check the marker mapping against the R script

`check_mapping()` raises if `5_5_compute_kdm_ba.R` no longer uses the panel
this notebook assumes.

In [ ]:
def _wz(x, w):
    """z-standardise on the survey-weighted mean and SD, as Mplus does."""
    x = np.asarray(x, float)
    w = np.asarray(w, float)
    mu = np.average(x, weights=w)
    sd = np.sqrt(np.average((x - mu) ** 2, weights=w))
    return (x - mu) / sd


def check_mapping():
    """Raise if the marker mapping no longer matches 5_5_compute_kdm_ba.R."""
    r = (BASE / 'scripts' / '5_5_compute_kdm_ba.R').read_text()
    missing = [f'{code} -> {name}' for code, name, _ in KDM_MARKERS
               if f'{name} ' not in r or code not in r]
    if missing:
        raise ValueError(
            'the KDM marker mapping in this notebook no longer matches '
            f'5_5_compute_kdm_ba.R: {missing}. Fix the mapping rather than '
            'the check -- a decomposition of the wrong panel is worse than none.')


def load():
    d = pd.read_csv(ANALYTIC, low_memory=False)
    missing = [k for k in RENAME if k not in d.columns]
    if missing:
        raise KeyError(f'{ANALYTIC.name} is missing {missing}')
    d = d.rename(columns=RENAME)
    return d[d.AGE >= 18].copy()


check_mapping()
d = load()
print(f'{len(d):,} adults; marker mapping matches 5_5_compute_kdm_ba.R')

## Step 2 — Fit every marker against every rhythm metric in every age cell

Holm correction across the 13 fitted markers within each rhythm metric and age cell.

In [ ]:
def fit_one(d, exposure, marker_code, lo, hi):
    need = [exposure, marker_code, 'WTMEC4YR'] + COVS
    s = d[(d.AGE >= lo) & (d.AGE < hi)]
    s = s[s[need].notna().all(axis=1)]
    if len(s) < 100:
        return None
    w = s['WTMEC4YR'].to_numpy(float)
    y = _wz(s[marker_code], w)
    X = np.column_stack([_wz(s[exposure], w)] +
                        [s[c].to_numpy(float) for c in COVS])
    r = wls_fit(X, y, w=w, se='hc3', add_const=True)
    # Index 1 is the exposure; index 0 is the intercept that add_const inserts.
    # The p value and interval come from wls_fit's exact-t distribution, as in
    # scripts 7 and 8.
    i = 1
    return dict(n=int(r['n']), beta=float(r['params'][i]), se=float(r['se'][i]),
                ci_lo=float(r['ci_lo'][i]), ci_hi=float(r['ci_hi'][i]),
                p=float(r['p'][i]))


def build():
    check_mapping()
    d = load()
    rows = []
    for cell_, lo, hi in CELLS:
        for exp in EXPOSURES:
            block = []
            for code, name, group in KDM_MARKERS + SII_INPUTS:
                if code not in d.columns:
                    raise KeyError(f'{code} is not in {ANALYTIC.name}')
                r = fit_one(d, exp, code, lo, hi)
                if r is None:
                    continue
                block.append(dict(age_cell=cell_, exposure=exp, code=code,
                                  marker=name, group=group, **r))
            if block:
                ph = holm(np.array([b['p'] for b in block]))
                for b, q in zip(block, ph):
                    b['p_holm'] = float(q)
                rows += block
    return pd.DataFrame(rows)


t = build()
print(f'{len(t)} fits across {t.exposure.nunique()} rhythm metrics, '
      f'{t.age_cell.nunique()} age cells, {t.marker.nunique()} markers')
t.head(13)

## Step 3 — Decomposition of log SII into its components

Refits log platelets, log neutrophils, log lymphocytes and log SII on each
rhythm metric with the same sample and covariates (outcomes on the raw log
scale), and reconstructs the log SII coefficient as
B(log PLT) + B(log NEU) − B(log LYM). The cell raises `ArithmeticError` if the
discrepancy exceeds 1e-6.

In [ ]:
def decompose_sii(d):
    rows = []
    for cell_, lo, hi in CELLS:
        for exp in EXPOSURES:
            need = ([exp, 'WTMEC4YR', 'LBXPLTSI', 'LBDNENO', 'LBDLYMNO',
                     'SII_log'] + COVS)
            s = d[(d.AGE >= lo) & (d.AGE < hi)]
            s = s[s[need].notna().all(axis=1)]
            s = s[(s.LBXPLTSI > 0) & (s.LBDNENO > 0) & (s.LBDLYMNO > 0)]
            if len(s) < 100:
                continue
            w = s['WTMEC4YR'].to_numpy(float)
            X = np.column_stack([_wz(s[exp], w)] +
                                [s[c].to_numpy(float) for c in COVS])

            def b(y):
                return float(wls_fit(X, y, w=w, se='hc3',
                                     add_const=True)['params'][1])

            b_plt = b(np.log(s.LBXPLTSI.to_numpy(float)))
            b_neu = b(np.log(s.LBDNENO.to_numpy(float)))
            b_lym = b(np.log(s.LBDLYMNO.to_numpy(float)))
            b_sii = b(s.SII_log.to_numpy(float))
            recon = b_plt + b_neu - b_lym
            rows.append(dict(age_cell=cell_, exposure=exp, n=len(s),
                             b_log_platelets=b_plt, b_log_neutrophils=b_neu,
                             b_log_lymphocytes=b_lym,
                             reconstructed=recon, fitted_log_SII=b_sii,
                             discrepancy=recon - b_sii))
    tt = pd.DataFrame(rows)
    worst = tt.discrepancy.abs().max()
    if worst > 1e-6:
        raise ArithmeticError(
            f'log SII does not reconstruct from its components (max '
            f'discrepancy {worst:.2e}). The identity log SII = log PLT + '
            f'log NEU - log LYM must hold exactly; a mismatch means the '
            f'samples or the transforms differ between the four fits.')
    return tt


dec = decompose_sii(load())
print('Why log SII is flat: the components cancel')
print(f'  {"cell":6s} {"metric":6s} {"logPLT":>8s} {"logNEU":>8s} '
      f'{"logLYM":>8s} {"= logSII":>9s}   {"check":>9s}')
for _, r in dec[dec.age_cell != '18+'].iterrows():
    print(f'  {r.age_cell:6s} {r.exposure:6s} {r.b_log_platelets:+8.4f} '
          f'{r.b_log_neutrophils:+8.4f} {r.b_log_lymphocytes:+8.4f} '
          f'{r.fitted_log_SII:+9.4f}   {r.discrepancy:9.1e}')
print('  identity log SII = log PLT + log NEU - log LYM holds exactly in every cell')

## Step 4 — Write Supplementary Table S16

- **Panel A** — 13 markers per rhythm metric per age cell. Coefficients are
  fully standardised β: both the rhythm metric and the marker are z-scored.
- **Panel B** — the log SII decomposition, one row per age cell × rhythm
  metric (nine rows). Coefficients are unstandardised B, the change in log
  units of the outcome per 1 s.d. of the rhythm metric, because the identity
  log SII = log PLT + log NEU − log LYM holds on the raw log scale and not for
  standardised β.

The 18+ rows are the secondary age-unstratified analysis and are placed last
in both panels.

In [ ]:
# Display names for the table.
MARKER_LABEL = {
    'albumin': 'Albumin', 'lymph': 'Lymphocyte percentage',
    'mcv': 'Mean corpuscular volume', 'glucose_mmol': 'Glucose',
    'rdw': 'Red cell distribution width', 'creat': 'Creatinine',
    'alp': 'Alkaline phosphatase', 'wbc': 'White blood cell count',
    'hba1c': 'Glycated haemoglobin (HbA1c)', 'totchol': 'Total cholesterol',
    'neutrophil count': 'Neutrophil count',
    'lymphocyte count': 'Lymphocyte count',
    'platelet count': 'Platelet count',
}
GROUP_LABEL = {
    'KDM panel: metabolic/organ': 'KDM: metabolic/organ',
    'KDM panel: immune': 'KDM: immune',
    'KDM panel: haematological': 'KDM: haematological',
    'SII input (not in KDM panel)': 'SII component',
    'SII input (lymph % is in KDM panel)': 'SII component†',
}
# AGE_LABEL comes from nature_palette; the pooled 18+ cell gets a longer
# label marking it as secondary.
AGE_LABEL = dict(AGE_LABEL, **{'18+': '≥18 (secondary, age-unstratified)'})
assert set(MARKER_LABEL) >= set(t.marker), 'a marker has no display name'
assert set(GROUP_LABEL) >= set(t.group), 'a marker group has no display name'


def summarise(t):
    """Count of Holm-significant associations per marker group in the two age cells."""
    out = []
    core = t[t.age_cell != '18+']
    imm = core[core.group.str.contains('immune')]
    met = core[core.group.str.contains('metabolic')]
    hae = core[core.group.str.contains('haematological')]
    sii = core[core.group.str.startswith('SII input')]
    for lab, g in [('KDM panel, immune (lymph, wbc)', imm),
                   ('KDM panel, metabolic/organ', met),
                   ('KDM panel, haematological', hae),
                   ('SII inputs', sii)]:
        out.append(f'{lab}: {int((g.p_holm < 0.05).sum())} of {len(g)} '
                   f'associations survive Holm across the two age cells '
                   f'(|beta| up to {g.beta.abs().max():.3f})')
    return out


# Panel A
panel_a = t.copy()
panel_a['β (95% CI)'] = [fmt_ci(r.beta, r.ci_lo, r.ci_hi)
                         for _, r in panel_a.iterrows()]
panel_a['P'] = panel_a.p.apply(lambda p: '<0.001' if p < 0.001 else f'{p:.3f}')
panel_a['P (Holm)'] = panel_a.p_holm.apply(
    lambda p: '<0.001' if p < 0.001 else f'{p:.3f}')
panel_a['Age group'] = panel_a.age_cell.map(AGE_LABEL)
panel_a['Marker'] = panel_a.marker.map(MARKER_LABEL)
panel_a['Marker group'] = panel_a.group.map(GROUP_LABEL)
# The NHANES variable column identifies each marker in the public NHANES files.
panel_a = panel_a[['Age group', 'exposure', 'Marker', 'Marker group', 'code',
                   'n', 'β (95% CI)', 'P', 'P (Holm)']].rename(
    columns={'exposure': 'Rhythm metric', 'code': 'NHANES variable'})

# Panel B
panel_b = dec.copy()
for c in ['b_log_platelets', 'b_log_neutrophils', 'b_log_lymphocytes',
          'reconstructed', 'fitted_log_SII']:
    panel_b[c] = panel_b[c].map(lambda v: fmt_est(v, 4))
panel_b['discrepancy'] = dec.discrepancy.map(lambda v: f'{v:.1e}')
panel_b['Age group'] = panel_b.age_cell.map(AGE_LABEL)
panel_b = panel_b[['Age group', 'exposure', 'n', 'b_log_platelets',
                   'b_log_neutrophils', 'b_log_lymphocytes', 'reconstructed',
                   'fitted_log_SII', 'discrepancy']].rename(columns={
    'exposure': 'Rhythm metric',
    'b_log_platelets': 'B, log platelet count',
    'b_log_neutrophils': 'B, log neutrophil count',
    'b_log_lymphocytes': 'B, log lymphocyte count',
    'reconstructed': 'Reconstructed B, log SII',
    'fitted_log_SII': 'Directly estimated B, log SII',
    'discrepancy': 'Difference'})
assert len(panel_b) == len(CELLS) * len(EXPOSURES) == 9, 'Panel B should be 9 rows'

NOTES = [
    'Supplementary Table S16 | Marker-level associations of rest-activity '
    'rhythm metrics with components of KDM biological age and the systemic '
    'immune-inflammation index.',
    '',
    'PANEL A. Marker-level associations. Survey-weighted least squares '
    '(WTMEC4YR = WTMEC2YR / 2) with HC3 standard errors. One model per marker '
    'per rhythm metric per age cell.',
    'Both the rhythm metric and the marker are z-standardised on the '
    'survey-weighted mean and standard deviation of the estimation sample, so '
    'beta is the change in marker standard deviations per 1 s.d. of the '
    'rhythm metric and is comparable across markers measured on different '
    'natural scales.',
    'Covariates: age, sex, body mass index, education, accelerometer wear '
    'time, self-reported sleep duration -- the same set as the main models.',
    'P (Holm): Holm-Bonferroni across the thirteen fitted markers within each '
    'rhythm metric and age cell. "Which marker carries the association" is one '
    'question asked thirteen times, so the thirteen are the family.',
    'The first ten markers are the KDM biological-age panel, in the order used '
    'by 5_5_compute_kdm_ba.R. The last three are the inputs to the systemic '
    'immune-inflammation index. † Lymphocytes enter the KDM panel as a '
    'percentage (LBXLYPCT) and the index as a count (LBDLYMNO); both are '
    'fitted because they are not the same variable.',
    'The KDM panel and the immune-inflammation index share inputs: lymphocytes '
    'appear in both, and neutrophils are the largest component of the white '
    'cell count that the KDM panel contains. The contrast drawn in the main '
    'text is therefore between two ways of combining overlapping markers, not '
    'between immune and non-immune biology.',
    'Rows labelled "≥18 (secondary, age-unstratified)" pool the full '
    'adult sample and are listed after the age-stratified cells; they are a '
    'secondary analysis and are not the basis of any claim in the main text.',
    '',
    'PANEL B. Algebraic decomposition of log SII. One row per age cell and '
    'rhythm metric.',
    'Because SII = platelets x neutrophils / lymphocytes, log SII = '
    'log(platelets) + log(neutrophils) - log(lymphocytes), and the coefficient '
    'on log SII is exactly that combination of the coefficients on the three '
    'logged components when all four are fitted on the same sample with the '
    'same covariates. The Difference column is the reconstructed value minus '
    'the directly fitted one; it is zero to machine precision by construction '
    'and is reported as a check on the computation, not as a result.',
    'Panel B reports B, NOT the standardised beta of Panel A, and the two are '
    'not interchangeable. The identity above holds for coefficients on the raw '
    'log scale; standardised coefficients each carry a different marker '
    'standard deviation in the denominator and would not sum to the '
    'coefficient on log SII. In Panel B the rhythm metric is z-standardised '
    'but the outcome is left on the log scale, so B is the change in log units '
    'of the component per 1 s.d. of the rhythm metric.',
    'The decomposition shows why the index is flat while its inputs are not. '
    'Weaker day-night contrast goes with fewer circulating neutrophils AND '
    'fewer lymphocytes. The index divides one by the other, so two '
    'associations of the same sign largely cancel.',
    'Consequently the null for this index should not be read as "rest-'
    'activity rhythm is unrelated to systemic immune state". It supports the '
    'narrower statement that rhythm is related to the abundance of '
    'circulating immune cells but not to the neutrophil-to-lymphocyte balance '
    'that the index is constructed to capture. Any ratio-based index built '
    'from these same counts, including the neutrophil-to-lymphocyte ratio, '
    'would be expected to behave the same way.',
    '',
    'BOTH PANELS. These are descriptive decompositions. They locate the '
    'association within the composite; they do not model the markers as '
    'mediators, and no causal ordering among them is implied. Weights enter '
    'the point estimate but the variance carries no stratum or '
    'primary-sampling-unit adjustment, so these standard errors are smaller '
    'than the design-based ones in the main models.',
]

out = SUPP / 'SuppTableS16_KDM_Component_Specificity.xlsx'
with pd.ExcelWriter(out, engine='openpyxl') as xw:
    panel_a.to_excel(xw, sheet_name='S16 Panel A markers', index=False)
    panel_b.to_excel(xw, sheet_name='S16 Panel B log SII', index=False)
    pd.DataFrame({'Notes': NOTES + [''] + summarise(t)}).to_excel(
        xw, sheet_name='Notes', index=False)
    for ws in xw.book.worksheets:
        for col in ws.columns:
            w = max((len(str(c.value)) for c in col if c.value is not None),
                    default=10)
            ws.column_dimensions[col[0].column_letter].width = min(
                max(w + 2, 10), 48)
        ws.freeze_panes = 'A2'
panel_a.to_csv(out.with_suffix('.csv'), index=False)
print(f'wrote {out.relative_to(BASE)}')
print(f'  Panel A: {len(panel_a)} rows   Panel B: {len(panel_b)} rows')

## Step 5 — Summary

Counts of Holm-significant associations per marker group, and the markers
significant in both the 18-59 and 60+ cells for each metric.

In [ ]:
for line in summarise(t):
    print(' ', line)

print('\nMarkers surviving Holm in BOTH the 18-59 and 60+ cells, by metric:')
core = t[t.age_cell != '18+']
for exp in EXPOSURES:
    g = core[(core.exposure == exp) & (core.p_holm < 0.05)]
    both = sorted({m for m in g.marker if (g.marker == m).sum() == 2})
    print(f'  {exp}: {both or "none"}')